Strategi yang dibandingkan:

1. **Equally Weighted (EW)** — Portofolio naif 1/N, bobot seragam untuk semua aset.
2. **Classical Markowitz (CM)** — Optimasi Mean-Variance tradisional dengan minimasi varians portofolio.
3. **Glasso Markowitz (GM)** — Markowitz dengan regularisasi *Graphical Lasso* pada matriks kovarians.
4. **Network Markowitz (NW)** — Integrasi RMT dan MST ke dalam fungsi objektif Markowitz, dengan parameter penalti $\gamma$.
   * **Statis:** $\gamma = 0$ (tanpa penalti jaringan), $\gamma = 1.0$, dan $\gamma = 2.0$
   * **Adaptif (Grid Search):** Optimasi $\gamma$ secara dinamis berbasis data latih historis. Meliputi **NW (Risk GS)** untuk minimasi risiko dan **NW (Return GS)** untuk maksimalisasi return.

---
## Sel 1 — Persiapan Library

Mengimpor semua library yang diperlukan untuk analisis jaringan, optimasi portofolio, dan visualisasi.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from scipy.optimize import minimize
from scipy.sparse.csgraph import minimum_spanning_tree
from scipy.linalg import eigh
from sklearn.covariance import GraphicalLassoCV
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries imported successfully!")

---
## Sel 2 — Memuat Data

Memuat data return dan harga dari file Excel. Fokus pada dataset return harian untuk **10 aset kripto utama** dalam periode **14 September 2017 – 17 Oktober 2019**.

In [ ]:
# Load data
excel_file = 'crypto_data_real.xlsx'
df_returns = pd.read_excel(excel_file, sheet_name='Returns', index_col=0)
df_prices  = pd.read_excel(excel_file, sheet_name='Prices',  index_col=0)

crypto_names = df_returns.columns.tolist()
n_assets     = len(crypto_names)

print(f"Data loaded: {df_returns.shape[0]} days, {n_assets} assets")
print(f"Period: {df_returns.index[0]} to {df_returns.index[-1]}")

---
## Sel 3 — Statistik Ringkasan (Summary Statistics)

Tabel statistik deskriptif mencakup Mean, Standar Deviasi, Kurtosis, dan Skewness untuk setiap aset kripto (mereplikasi **Tabel 1** pada paper acuan).

In [ ]:
# Create summary statistics table
summary_stats = pd.DataFrame({
    'Mean':     df_returns.mean(),
    'Std':      df_returns.std(),
    'Kurtosis': df_returns.kurt(),
    'Skewness': df_returns.skew()
})

print("TABLE 1 | Summary statistics.")
print(summary_stats.round(4))

**Analisis Tabel 1:** Data menunjukkan bahwa sebagian besar aset kripto memiliki nilai *kurtosis* yang sangat tinggi (di atas 3), terutama TRX (22.18) dan XRP (18.90). Hal ini mengonfirmasi sifat *heavy-tailed* dari distribusi return kripto, di mana risiko ekstrem lebih sering terjadi dibandingkan distribusi normal. Nilai *Mean* yang mendekati nol mencerminkan efisiensi pasar secara rata-rata, namun volatilitas (*Std*) yang tinggi (rata-rata > 4%) menunjukkan risiko investasi yang sangat besar.

---
## Sel 4 — Visualisasi Harga Ternormalisasi (Figure 1 & 2)

Mereplikasi *Normalized cryptocurrency price series* dengan mengatur harga awal setiap aset menjadi **100** pada tanggal **7 Januari 2018**.

In [ ]:
# --- Figure 1: BTC, ETH, USDT, BCH, LTC ---
assets_1 = ['BTC', 'ETH', 'USDT', 'BCH', 'LTC']
df_norm_1 = (df_prices.loc['2018-01-07':, assets_1] /
             df_prices.loc['2018-01-07', assets_1]) * 100

colors_1 = ['black', 'red', 'green', 'blue', 'cyan']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_1.columns):
    plt.plot(df_norm_1.index, df_norm_1[col],
             label=col, color=colors_1[i])
plt.title('Figure 1 | Normalized Price Series I (BTC, ETH, USDT, BCH, LTC)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

# --- Figure 2: XRP, BNB, EOS, XLM, TRX ---
assets_2 = ['XRP', 'BNB', 'EOS', 'XLM', 'TRX']
df_norm_2 = (df_prices.loc['2018-01-07':, assets_2] /
             df_prices.loc['2018-01-07', assets_2]) * 100

colors_2 = ['magenta', 'gold', 'lightgrey', 'black', 'red']

plt.figure(figsize=(12, 5))
for i, col in enumerate(df_norm_2.columns):
    plt.plot(df_norm_2.index, df_norm_2[col],
             label=col, color=colors_2[i])
plt.title('Figure 2 | Normalized Price Series II (XRP, BNB, EOS, XLM, TRX)')
plt.ylabel('Normalized Price (base=100)')
plt.legend()
plt.tight_layout()
plt.show()

---
## Sel 5 — Visualisasi Minimum Spanning Tree (Figure 3 & 4)

Mereplikasi Figure 3 dari paper acuan, menunjukkan struktur jaringan aset kripto selama dua periode berbeda:
- **Figure 3**: Periode gelembung spekulatif (Sep 2017 – Jan 2018)
- **Figure 4**: Periode stabil (Jun 2019 – Okt 2019)

In [ ]:
# --- Figure 3 | MST Speculative Bubble Period ---
df_bubble   = df_returns.loc['2017-09-14':'2018-01-31']
corr_bubble = df_bubble.corr()
dist_bubble = np.sqrt(2 * (1 - corr_bubble))

G          = nx.from_pandas_adjacency(dist_bubble)
mst_bubble = nx.minimum_spanning_tree(G, weight='weight')

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(mst_bubble, seed=42)
nx.draw(mst_bubble, pos, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 3 | MST September 2017 - January 2018', fontsize=12)
plt.show()

# --- Figure 4 | MST Stable Period ---
df_stable   = df_returns.loc['2019-06-01':'2019-10-17']
corr_stable = df_stable.corr()
dist_stable = np.sqrt(2 * (1 - corr_stable))

G2         = nx.from_pandas_adjacency(dist_stable)
mst_stable = nx.minimum_spanning_tree(G2, weight='weight')

plt.figure(figsize=(10, 8))
pos2 = nx.spring_layout(mst_stable, seed=42)
nx.draw(mst_stable, pos2, with_labels=True, node_color='orange',
        node_size=1500, edge_color='black', linewidths=1.5,
        font_size=10, font_weight='bold')
plt.title('Figure 4 | MST June 2019 - October 2019', fontsize=12)
plt.show()

**Analisis Figure 3 & 4 (Evolusi Topologi Minimum Spanning Tree):**

Perbandingan antara *Figure 3* (Fase *Speculative Bubble*, Sep 2017 – Jan 2018) dan *Figure 4* (Fase *Stable*, Jun 2019 – Okt 2019) memperlihatkan perubahan signifikan dalam struktur ketergantungan antar aset kripto:

1. **Fase Speculative Bubble (Figure 3):** Struktur MST cenderung lebih memanjang dan berantai (*chain-like topology*). Tidak ada satu aset pun yang mendominasi sebagai pusat (*hub*) absolut. Peran sentralitas terbagi antara **ETH** dan **LTC** yang bertindak sebagai jembatan antar klaster aset. Hal ini mengindikasikan bahwa selama periode euforia, pergerakan harga cenderung lebih terfragmentasi dan setiap altcoin memiliki dinamika spekulatifnya masing-masing.

2. **Fase Stable (Figure 4):** Struktur jaringan berubah drastis menjadi sangat terpusat (*star-like topology*). **ETH** muncul sebagai titik pusat (*super hub*) yang mendominasi jaringan dan menghubungkan secara langsung 5 aset lainnya (LTC, BTC, BNB, TRX, BCH). Hal ini menandakan bahwa pada kondisi pasar yang lebih tenang secara fundamental, pergerakan aset lebih seragam (*highly coupled*) dan berkiblat pada satu atau dua pemimpin pasar, sehingga risiko sistemik terkonsentrasi di pusat jaringan.

Perubahan dari struktur berantai yang terdesentralisasi menjadi struktur bintang yang tersentralisasi inilah yang ditangkap oleh sentralitas vektor eigen (*eigenvector centrality*) pada fungsi penalti **Network Markowitz (NW)**, memungkinkannya meminimalisir eksposur pada *super hub* yang rentan kolaps bersistem.

---
## Sel 6 — Fungsi Pembantu (Helper Functions)

Implementasi fungsi-fungsi pembantu yang digunakan oleh semua strategi:
- **RMT Filter** — Penyaringan noise pada matriks korelasi menggunakan Random Matrix Theory (batas Marchenko-Pastur λ+).
- **MST Builder** — Konstruksi matriks jarak dari korelasi: $d_{ij} = \sqrt{2(1 - \rho_{ij})}$.
- **Eigenvector Centrality** — Mengukur sentralitas aset dalam jaringan.
- **Metrik Risiko** — VaR, Rachev Ratio, Maximum Drawdown.


In [ ]:
def apply_rmt_filter(returns_data):
    """Filter noise dari matriks korelasi menggunakan Random Matrix Theory."""
    if isinstance(returns_data, pd.DataFrame):
        data = returns_data.values
    else:
        data = returns_data
    T, N = data.shape
    Q    = T / N
    C    = np.corrcoef(data.T)

    eigenvalues, eigenvectors = eigh(C)
    eigenvalues  = eigenvalues[::-1]
    eigenvectors = eigenvectors[:, ::-1]

    # Marchenko-Pastur upper bound
    lambda_plus      = 1 + (1/Q) + 2*np.sqrt(1/Q)
    significant_mask = eigenvalues > lambda_plus

    Lambda_filtered = np.diag(np.where(significant_mask, eigenvalues, 0))
    C_filtered      = eigenvectors @ Lambda_filtered @ eigenvectors.T
    return C_filtered


def build_mst(correlation_matrix):
    """Hitung matriks jarak dari matriks korelasi."""
    distance_matrix = np.sqrt(2 - 2*correlation_matrix)
    np.fill_diagonal(distance_matrix, 0)
    return distance_matrix


def compute_eigenvector_centrality(distance_matrix):
    """Hitung eigenvector centrality dari matriks jarak."""
    adjacency = 1 / (distance_matrix + 1e-8)
    np.fill_diagonal(adjacency, 0)
    eigenvalues, eigenvectors = eigh(adjacency)
    principal_eigenvector = np.abs(eigenvectors[:, -1])
    centrality = principal_eigenvector / principal_eigenvector.sum()
    return centrality


def calculate_var(returns, confidence=0.95):
    """Hitung Value at Risk."""
    return np.percentile(returns, (1 - confidence) * 100)


def calculate_rachev_ratio(returns, alpha=0.10):
    """Hitung Rachev Ratio: CVaR_upper / CVaR_lower."""
    threshold_upper = np.percentile(returns, (1 - alpha) * 100)
    threshold_lower = np.percentile(returns, alpha * 100)
    cvar_upper = returns[returns >= threshold_upper].mean()
    cvar_lower = abs(returns[returns <= threshold_lower].mean())
    return cvar_upper / cvar_lower if cvar_lower > 0 else 0


def calculate_max_drawdown(cumulative_returns):
    """Hitung Maximum Drawdown dari cumulative returns."""
    running_max = np.maximum.accumulate(cumulative_returns)
    drawdown    = (cumulative_returns - running_max) / running_max
    return drawdown.min()


def get_assets_graph_diversify(returns_window, corr_threshold=0.4):
    """Mencari aset untuk diversifikasi menggunakan Maximum Independent Set."""
    corr_mat = returns_window.corr()
    G        = nx.Graph()
    assets   = list(returns_window.mean()
                    .sort_values(ascending=False).index)
    G.add_nodes_from(assets)
    for i, a1 in enumerate(assets):
        for a2 in assets[i+1:]:
            if abs(corr_mat.loc[a1, a2]) > corr_threshold:
                G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))


def get_assets_graph_diversify_rmt(returns_window, corr_threshold=0.4):
    """Mencari aset independen menggunakan korelasi yang sudah difilter RMT."""
    assets = returns_window.columns.tolist()
    corr_f = apply_rmt_filter(returns_window)
    G      = nx.Graph()
    G.add_nodes_from(assets)
    for i, a1 in enumerate(assets):
        for j, a2 in enumerate(assets):
            if i < j and abs(corr_f[i, j]) > corr_threshold:
                G.add_edge(a1, a2)
    return list(nx.approximation.maximum_independent_set(G))


print("Helper functions defined successfully!")

---
## Sel 7 — Analisis Dinamika MST (Figure 5)

Analisis *rolling window* untuk menghitung dua metrik dinamis jaringan: **Max Link Distance** dan **Residuality**.

In [ ]:
def calculate_rolling_mst_metrics(returns_df, window=120):
    """Hitung dinamika MST dengan rolling window."""
    dates          = returns_df.index[window:]
    max_links      = []
    residualities  = []
    for i in range(window, len(returns_df)):
        window_data = returns_df.iloc[i - window:i]
        corr_f      = apply_rmt_filter(window_data)
        mst_weights = build_mst(corr_f)
        max_links.append(np.max(mst_weights))
        residualities.append(
            np.sum(mst_weights) / (returns_df.shape[1] - 1)
        )
    return pd.DataFrame(
        {'Max Link': max_links, 'Residuality': residualities},
        index=dates
    )


mst_dyn = calculate_rolling_mst_metrics(df_returns)

# Visualisasi dengan dual axis
fig, ax1 = plt.subplots(figsize=(12, 7))
ax1.plot(mst_dyn.index, mst_dyn['Max Link'],
         color='black', label='Max Link')
ax1.set_ylabel('Max Link Distance', color='black')
ax1.set_xlabel('Date')

ax2 = ax1.twinx()
ax2.plot(mst_dyn.index, mst_dyn['Residuality'],
         color='red', label='Residuality')
ax2.set_ylabel('Residuality', color='red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Figure 5 | MST Dynamics: Max Link & Residuality over Time')
plt.tight_layout()
plt.show()

---
## Sel 8 — Implementasi Strategi Portofolio

Setiap strategi diimplementasikan sebagai kelas Python yang mewarisi `PortfolioStrategy`.



In [ ]:
class PortfolioStrategy:
    def __init__(self, name):
        self.name            = name
        self.weights_history = []
        self.returns_history = []
    def get_weights(self, returns_data):
        raise NotImplementedError


# -- 1. Equally Weighted -----------------------------------------------------
class EquallyWeighted(PortfolioStrategy):
    def get_weights(self, returns_data):
        n = returns_data.shape[1]
        return np.ones(n) / n


# -- 2. Classical Markowitz --------------------------------------------------
class ClassicalMarkowitz(PortfolioStrategy):
    """Optimasi Mean-Variance tradisional (minimasi varians portofolio)."""
    def get_weights(self, returns_data):
        n_assets = returns_data.shape[1]
        mu = returns_data.mean().values
        S  = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP',
                       bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets


# -- 3. Glasso Markowitz -----------------------------------------------------
class GlassoMarkowitz(PortfolioStrategy):
    """Markowitz dengan matriks presisi yang diestimasi via Graphical Lasso."""
    def get_weights(self, returns_data):
        n_assets = returns_data.shape[1]
        mu = returns_data.mean().values
        try:
            glasso = GraphicalLassoCV()
            glasso.fit(returns_data.values)
            S = glasso.covariance_
        except Exception:
            S = returns_data.cov().values
        objective   = lambda w: w @ S @ w
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP',
                       bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets


# -- 4. Network Markowitz ----------------------------------------------------
class NetworkMarkowitz(PortfolioStrategy):
    """Network Markowitz: RMT + MST + penalti sentralitas eigenvector."""
    def __init__(self, name="Network Markowitz", gamma=0):
        super().__init__(name)
        self.gamma = gamma

    def get_weights(self, returns_data):
        n_assets = returns_data.shape[1]
        mu   = returns_data.mean().values
        sig  = returns_data.std().values
        Cf   = apply_rmt_filter(returns_data)
        dist = build_mst(Cf)
        cent = compute_eigenvector_centrality(dist)
        Sf   = np.outer(sig, sig) * Cf
        objective   = lambda w: w @ Sf @ w + self.gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP',
                       bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets




In [ ]:
class OptimizedGridNW(PortfolioStrategy):
    """Network Markowitz dengan optimasi parameter Gamma via Grid Search."""
    def __init__(self, name="NW (Grid Search)"):
        super().__init__(name)
        self.best_gamma = 0.5
        self.param_history = []
        self.grid_performance_history = []

    def _get_weights_logic(self, data, gamma):
        n_assets = data.shape[1]
        mu   = data.mean().values
        sig  = data.std().values
        Cf   = apply_rmt_filter(data)
        dist = build_mst(Cf)
        cent = compute_eigenvector_centrality(dist)
        Sf   = np.outer(sig, sig) * Cf
        objective   = lambda w: w @ Sf @ w + gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP',
                       bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets

    def get_weights(self, returns_data):
        split = int(0.8 * len(returns_data))
        v_train, v_val = returns_data.iloc[:split], returns_data.iloc[split:]
        
        gamma_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
        min_var    = float('inf')
        
        for g in gamma_grid:
            w = self._get_weights_logic(v_train, g)
            val_risk = v_val.dot(w).var() 
            
            self.grid_performance_history.append({
                'date': returns_data.index[-1],
                'gamma': g,
                'risk': val_risk
            })
            
            if val_risk < min_var:
                min_var = val_risk
                self.best_gamma = g
        
        self.param_history.append({
            'date': returns_data.index[-1],
            'gamma': self.best_gamma
        })
        
        return self._get_weights_logic(returns_data, self.best_gamma)

class OptimizedReturnNW(PortfolioStrategy):
    """Network Markowitz dengan optimasi parameter Gamma via Grid Search (Max Return)."""
    def __init__(self, name="NW (Return GS)"):
        super().__init__(name)
        self.best_gamma = 0.5
        self.param_history = []
        self.grid_performance_history = []

    def _get_weights_logic(self, data, gamma):
        n_assets = data.shape[1]
        mu   = data.mean().values
        sig  = data.std().values
        Cf   = apply_rmt_filter(data)
        dist = build_mst(Cf)
        cent = compute_eigenvector_centrality(dist)
        Sf   = np.outer(sig, sig) * Cf
        objective   = lambda w: w @ Sf @ w + gamma * np.sum(cent * w)
        constraints = [
            {'type': 'eq',   'fun': lambda w: np.sum(w) - 1},
            {'type': 'ineq', 'fun': lambda w: w @ mu - mu.mean()}
        ]
        res = minimize(objective, np.ones(n_assets) / n_assets,
                       method='SLSQP',
                       bounds=[(0, 1)] * n_assets,
                       constraints=constraints)
        return res.x if res.success else np.ones(n_assets) / n_assets

    def get_weights(self, returns_data):
        split = int(0.8 * len(returns_data))
        v_train, v_val = returns_data.iloc[:split], returns_data.iloc[split:]
        
        gamma_grid = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0]
        max_ret    = -float('inf')
        
        for g in gamma_grid:
            w = self._get_weights_logic(v_train, g)
            val_ret = (v_val.dot(w)).sum() 
            
            self.grid_performance_history.append({
                'date': returns_data.index[-1],
                'gamma': g,
                'return': val_ret
            })
            
            if val_ret > max_ret:
                max_ret = val_ret
                self.best_gamma = g
        
        self.param_history.append({
            'date': returns_data.index[-1],
            'gamma': self.best_gamma
        })
        
        return self._get_weights_logic(returns_data, self.best_gamma)



---
## Sel 9 — Framework Backtesting

Sistem pengujian menggunakan *rolling window* dengan parameter:
- **Window size**: 120 hari
- **Rebalance frequency**: 7 hari
- **Transaction cost**: 0.1% (10 basis points)

In [ ]:
def backtest_strategy(strategy, df_returns,
                      window_size=120,
                      rebalance_freq=7,
                      transaction_cost=0.001):
    """Simulasi backtesting dengan rolling window."""
    portfolio_returns = []
    dates             = []
    for i in range(window_size, len(df_returns), rebalance_freq):
        train    = df_returns.iloc[i - window_size:i]
        w        = strategy.get_weights(train)
        test_end = min(i + rebalance_freq, len(df_returns))
        test_data = df_returns.iloc[i:test_end]
        for j in range(len(test_data)):
            daily_ret = np.dot(w, test_data.iloc[j].values)
            if j == 0 and len(portfolio_returns) > 0:
                daily_ret -= transaction_cost
            portfolio_returns.append(daily_ret)
            dates.append(test_data.index[j])

    res_df = pd.DataFrame({'date': dates,
                           'return': portfolio_returns})
    res_df['cumulative_return'] = (1 + res_df['return']).cumprod()
    return {
        'strategy':           strategy.name,
        'returns':            np.array(portfolio_returns),
        'cumulative_returns': res_df['cumulative_return'].values,
        'results_df':         res_df
    }


print("Backtest function defined!")

---
## Sel 10 — Eksekusi Backtesting



In [ ]:
# --- Inisialisasi strategi ---
strategies = [
    EquallyWeighted("EW"),
    ClassicalMarkowitz("CM"),
    GlassoMarkowitz("GM"),
    NetworkMarkowitz("NW (gamma=0)", gamma=0),
    NetworkMarkowitz("NW (gamma=1.0)", gamma=1.0),
    NetworkMarkowitz("NW (gamma=2.0)", gamma=2.0),
    OptimizedGridNW("NW (Risk GS)"),
    OptimizedReturnNW("NW (Return GS)")
]

# --- Eksekusi backtest ---
results = {}
for strat in strategies:
    print(f"Running: {strat.name}...")
    results[strat.name] = backtest_strategy(strat, df_returns)

print("\nAll backtests completed!")

---
## Sel 9.6 — Rekapitulasi Parameter Optimal (Table 7)

Tabel di bawah menunjukkan log parameter optimal yang dipilih oleh mekanisme Grid Search pada periode-periode terakhir.

In [ ]:
# --- Visualisasi Hasil Grid Search (Heatmap & Pivot Table) ---
grid_strat = next((s for s in strategies if isinstance(s, OptimizedGridAGGP)), None)

if grid_strat and grid_strat.grid_performance_history:
    all_grid_df = pd.DataFrame(grid_strat.grid_performance_history)
    
    # 1. Pivot Table: Rata-rata Risiko per Kombinasi (S, T)
    pivot_risk = all_grid_df.pivot_table(index='sensitivity', columns='threshold', values='risk', aggfunc='mean')
    
    print("\nTABLE 8 | Average Validation Risk (Variance) per Parameter Combination")
    display(pivot_risk.style.background_gradient(cmap='RdYlGn_r').format("{:.6f}"))
    
    # 2. Pivot Table: Frekuensi Pemilihan Parameter Optimal
    best_params_df = pd.DataFrame(grid_strat.param_history)
    pivot_freq = best_params_df.pivot_table(index='sensitivity', columns='threshold', aggfunc='size', fill_value=0)
    
    print("\nTABLE 9 | Frequency of Optimal Parameter Selection")
    display(pivot_freq.style.background_gradient(cmap='Blues'))
    
    # Visualisasi Figure 7 yang lama tetap dipertahankan atau diganti dengan ini
    hist_df = pd.DataFrame(grid_strat.param_history).set_index('date')
    fig, ax1 = plt.subplots(figsize=(14, 5))
    ax1.step(hist_df.index, hist_df['sensitivity'], where='post', color='blue', label='Sensitivity')
    ax2 = ax1.twinx()
    ax2.step(hist_df.index, hist_df['threshold'], where='post', color='green', linestyle='--', label='Threshold')
    plt.title('FIGURE 7 | Tailoring Optimal Parameters over Time')
    plt.show()
    # 3. Analisis Hasil Grid Search untuk NW (Grid Search)
    nw_grid_strat = next((s for s in strategies if isinstance(s, OptimizedGridNW)), None)
    if nw_grid_strat and nw_grid_strat.grid_performance_history:
        nw_all_grid_df = pd.DataFrame(nw_grid_strat.grid_performance_history)
        nw_pivot_risk = nw_all_grid_df.pivot_table(index='gamma', values='risk', aggfunc='mean')
        print("\nTABLE 10 | Average Validation Risk per Gamma (NW (Grid Search))")
        display(nw_pivot_risk.style.background_gradient(cmap='RdYlGn_r').format("{:.6f}"))
        
        nw_hist_df = pd.DataFrame(nw_grid_strat.param_history).set_index('date')
        plt.figure(figsize=(14, 4))
        plt.step(nw_hist_df.index, nw_hist_df['gamma'], where='post', color='indigo', label='Optimal Gamma (NW)')
        plt.title('FIGURE 8 | Optimal Gamma Selection over Time (NW)')
        plt.ylabel('Gamma Value')
        plt.legend()
        plt.show()
else:
    print("No grid history found.")

    # 4. Analisis Hasil Grid Search untuk NW (Return GS)
    nw_ret_strat = next((s for s in strategies if isinstance(s, OptimizedReturnNW)), None)
    if nw_ret_strat and nw_ret_strat.grid_performance_history:
        nw_ret_grid_df = pd.DataFrame(nw_ret_strat.grid_performance_history)
        nw_pivot_ret = nw_ret_grid_df.pivot_table(index='gamma', values='return', aggfunc='mean')
        print("\nTABLE 11 | Average Validation Return per Gamma (NW - Return Optimization)")
        display(nw_pivot_ret.style.background_gradient(cmap='RdYlGn').format("{:.6f}"))
        
        hist_ret_df = pd.DataFrame(nw_ret_strat.param_history).set_index('date')
        plt.figure(figsize=(14, 4))
        plt.step(hist_ret_df.index, hist_ret_df['gamma'], where='post', color='orange', label='Optimal Gamma (Return GS)')
        plt.title('FIGURE 9 | Optimal Gamma Selection for Return over Time')
        plt.ylabel('Gamma Value')
        plt.legend()
        plt.show()

    # 5. Analisis Hasil Grid Search untuk AGGP (Return GS)
    aggp_ret_strat = next((s for s in strategies if isinstance(s, OptimizedReturnAGGP)), None)
    if aggp_ret_strat and aggp_ret_strat.grid_performance_history:
        aggp_ret_df = pd.DataFrame(aggp_ret_strat.grid_performance_history)
        aggp_piv_ret = aggp_ret_df.pivot_table(index='sensitivity', columns='threshold', values='return', aggfunc='mean')
        print("\nTABLE 12 | Average Validation Return for AGGP (Return Optimization)")
        display(aggp_piv_ret.style.background_gradient(cmap='RdYlGn'))
        
        hist_aggp_ret = pd.DataFrame(aggp_ret_strat.param_history).set_index('date')
        fig, ax1 = plt.subplots(figsize=(14, 4))
        ax1.step(hist_aggp_ret.index, hist_aggp_ret['sensitivity'], where='post', color='teal', label='Sensitivity ($s$)')
        ax2 = ax1.twinx()
        ax2.step(hist_aggp_ret.index, hist_aggp_ret['threshold'], where='post', color='salmon', label='Threshold ($t$)')
        plt.title('FIGURE 10 | Optimal AGGP Parameters over Time (Return GS)')
        ax1.set_ylabel('Sensitivity')
        ax2.set_ylabel('Threshold')
        ax1.legend(loc='upper left')
        ax2.legend(loc='upper right')
        plt.show()
    # 6. Analisis Hasil Grid Search untuk GD (Return GS)
    gd_ret_strat = next((s for s in strategies if isinstance(s, OptimizedReturnGD)), None)
    if gd_ret_strat and gd_ret_strat.grid_performance_history:
        gd_ret_df = pd.DataFrame(gd_ret_strat.grid_performance_history)
        gd_pivot_ret = gd_ret_df.pivot_table(index='theta', values='return', aggfunc='mean')
        print("\nTABLE 13 | Average Validation Return per Theta (GD - Return Optimization)")
        display(gd_pivot_ret.style.background_gradient(cmap='RdYlGn').format("{:.6f}"))
        
        hist_gd_df = pd.DataFrame(gd_ret_strat.param_history).set_index('date')
        plt.figure(figsize=(14, 4))
        plt.step(hist_gd_df.index, hist_gd_df['theta'], where='post', color='brown', label='Optimal Theta (GD)')
        plt.title('FIGURE 11 | Optimal Correlation Threshold Selection for Return over Time (GD)')
        plt.ylabel('Theta Value')
        plt.legend()
        plt.show()
    # 7. Analisis Hasil Grid Search untuk GD (Risk GS)
    gd_risk_strat = next((s for s in strategies if isinstance(s, OptimizedGridGD)), None)
    if gd_risk_strat and gd_risk_strat.grid_performance_history:
        gd_risk_df = pd.DataFrame(gd_risk_strat.grid_performance_history)
        gd_pivot_risk = gd_risk_df.pivot_table(index='theta', values='risk', aggfunc='mean')
        print("\nTABLE 14 | Average Validation Risk per Theta (GD - Risk Optimization)")
        display(gd_pivot_risk.style.background_gradient(cmap='RdYlGn_r').format("{:.8f}"))
        
        hist_gd_risk_df = pd.DataFrame(gd_risk_strat.param_history).set_index('date')
        plt.figure(figsize=(14, 4))
        plt.step(hist_gd_risk_df.index, hist_gd_risk_df['theta'], where='post', color='brown', label='Optimal Theta (Risk GS)')
        plt.title('FIGURE 12 | Optimal Correlation Threshold Selection for Minimum Risk over Time (GD)')
        plt.ylabel('Theta Value')
        plt.legend()
        plt.show()


---
## Sel 11 — Analisis Performa Periodik (Table 2)

Tabel perbandingan **Cumulative Profit/Loss** yang disampel setiap 4 bulan (Januari, Mei, September).

In [ ]:
target_dates = [
    '2018-01-31', '2018-05-31', '2018-09-30',
    '2019-01-31', '2019-05-31', '2019-09-30'
]

table2_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for td in target_dates:
        idx     = df_res.index.searchsorted(pd.Timestamp(td))
        idx     = min(idx, len(df_res) - 1)
        cum_ret = (df_res['cumulative_return'].iloc[idx] - 1) * 100
        row[td] = round(cum_ret, 2)
    table2_rows.append(row)

table2 = pd.DataFrame(table2_rows).set_index('Strategy')
table2.columns = ['Jan-2018', 'May-2018', 'Sep-2018',
                  'Jan-2019', 'May-2019', 'Sep-2019']

print("TABLE 2 | Cumulative Profits and Losses (%)")
display(table2)

**Analisis Tabel 2:** Tabel ini menunjukkan performa kerugian kumulatif selama rentang periode *crypto winter* (2018-2019). Terdapat beberapa temuan penting yang terlihat:

1. **Efektivitas Peningkatan Penalti (Gamma):** Pada model *Network Markowitz* (NW) dengan parameter statis, terlihat jelas bahwa penguatan dominasi struktur jaringan (MST) berkorelasi positif dengan membaiknya proteksi risiko. Tercatat pada Sep-2019, NW ($\gamma=0$) membukukan kerugian **-68.05%**. Ini membaik secara signifikan di NW ($\gamma=1.0$) sebesar **-48.05%**, dan paling tangguh di NW ($\gamma=2.0$) sebesar **-46.38%**.
2. **Daya Tahan Grid Search:** Secara keseluruhan hingga akhir periode observasi (Sep-2019), strategi **NW (Return GS)** mencatatkan proteksi *loss* yang sebanding dan bersaing ketat dengan NW ($\gamma=2.0$), menunjukkan bahwa optimasi parameter adaptif berbasis *rolling window* efektif. Hasil ini luar biasa jika dibandingkan dengan perburukan *drawdown* yang parah pada *baseline* naif (EW: **-88.82%**) dan optimasi Markowitz tradisional (CM: **-62.16%**).

---
## Sel 12 — Visualisasi Performa Kumulatif (Figure 6)

Evolusi nilai portofolio dengan asumsi **investasi awal 100 USD**.

In [ ]:
plt.figure(figsize=(14, 7))
for name, res in results.items():
    plt.plot(res['results_df']['date'],
             res['cumulative_returns'] * 100,
             label=name)
plt.title('FIGURE 6 | Performances of different portfolio strategies')
plt.xlabel('Date')
plt.ylabel('Portfolio Value (USD)')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## Sel 13 — Analisis Risiko Periodik (Table 3 — VaR 95%)

Value at Risk (VaR) dengan tingkat kepercayaan **95%** dihitung untuk jendela 4 bulan terakhir di setiap titik sampling. Nilai disajikan dalam bentuk absolut dikali faktor skala 100.

In [ ]:
period_ranges = [
    ('Jan-2018', '2017-10-01', '2018-01-31'),
    ('May-2018', '2018-02-01', '2018-05-31'),
    ('Sep-2018', '2018-06-01', '2018-09-30'),
    ('Jan-2019', '2018-10-01', '2019-01-31'),
    ('May-2019', '2019-02-01', '2019-05-31'),
    ('Sep-2019', '2019-06-01', '2019-09-30'),
]

table3_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, start, end in period_ranges:
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 0:
            v = abs(calculate_var(segment, 0.95)) * 100
        else:
            v = np.nan
        row[period_label] = round(v, 4)
    table3_rows.append(row)

table3 = pd.DataFrame(table3_rows).set_index('Strategy')
print("TABLE 3 | VaR 95% (4-month window, scaled x100)")
display(table3)

---
## Sel 14 — Analisis Risk-Adjusted Return (Table 4 — Sharpe Ratio)

In [ ]:
table4_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, start, end in period_ranges:
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 1 and np.std(segment) > 1e-6:
            sr = np.mean(segment) / np.std(segment)
        else:
            sr = np.nan
        row[period_label] = round(sr, 4)
    table4_rows.append(row)

table4 = pd.DataFrame(table4_rows).set_index('Strategy')
print("TABLE 4 | Sharpe Ratio (4-month window)")
display(table4)

**Analisis Tabel 2:** Tabel ini menunjukkan performa kerugian kumulatif selama rentang periode *crypto winter* (2018-2019). Terdapat beberapa temuan penting yang terlihat:

1. **Efektivitas Peningkatan Penalti (Gamma):** Pada model *Network Markowitz* (NW) dengan parameter statis, terlihat jelas bahwa penguatan dominasi struktur jaringan (MST) berkorelasi positif dengan membaiknya proteksi risiko. Tercatat pada Sep-2019, NW ($\gamma=0$) membukukan kerugian **-68.05%**. Ini membaik secara signifikan di NW ($\gamma=1.0$) sebesar **-48.05%**, dan paling tangguh di NW ($\gamma=2.0$) sebesar **-46.38%**.
2. **Daya Tahan Grid Search:** Secara keseluruhan hingga akhir periode observasi (Sep-2019), strategi **NW (Return GS)** mencatatkan proteksi *loss* yang sebanding dan bersaing ketat dengan NW ($\gamma=2.0$), menunjukkan bahwa optimasi parameter adaptif berbasis *rolling window* efektif. Hasil ini luar biasa jika dibandingkan dengan perburukan *drawdown* yang parah pada *baseline* naif (EW: **-88.82%**) dan optimasi Markowitz tradisional (CM: **-62.16%**).

---
## Sel 15 — Analisis Tail-Risk (Table 5 — Rachev Ratio)

Rachev Ratio (RR) memberikan wawasan tentang asimetri distribusi imbal hasil pada ekor (*tails*):

$$\text{RR} = \frac{\text{CVaR}_{\text{upper}}(10\%)}{\text{CVaR}_{\text{lower}}(10\%)}$$

In [ ]:
table5_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    row    = {'Strategy': strat_name}
    for period_label, start, end in period_ranges:
        segment = df_res.loc[start:end, 'return'].values
        if len(segment) > 5:
            rr = calculate_rachev_ratio(segment, alpha=0.10)
        else:
            rr = np.nan
        row[period_label] = round(rr, 4)
    table5_rows.append(row)

table5 = pd.DataFrame(table5_rows).set_index('Strategy')
print("TABLE 5 | Rachev Ratio (alpha=10%, 4-month window)")
display(table5)

---
## Sel 16 — Analisis Fase Pasar (Table 6)

Pemetaan performa ke tiga fase utama pasar:
- **Bearish**: Januari 2018 – Maret 2019
- **Recovery**: April 2019 – Juni 2019
- **Stable/Sideways**: Juli 2019 – Oktober 2019

In [ ]:
fase_pasar = {
    'Bearish':  ('2018-01-01', '2019-03-31'),
    'Recovery': ('2019-04-01', '2019-06-30'),
    'Stable':   ('2019-07-01', '2019-10-17')
}

sharpe_rows = []
rachev_rows = []
for strat_name, res in results.items():
    df_res = res['results_df'].set_index('date')
    sr_row = {'Strategy': strat_name}
    rr_row = {'Strategy': strat_name}
    for fase, (start, end) in fase_pasar.items():
        segment = df_res.loc[start:end, 'return'].values
        # Sharpe Ratio
        if len(segment) > 1 and np.std(segment) > 1e-6:
            sr = np.mean(segment) / np.std(segment)
        else:
            sr = np.nan
        sr_row[fase] = round(sr, 4)
        # Rachev Ratio
        if len(segment) > 5:
            rr = calculate_rachev_ratio(segment, alpha=0.10)
        else:
            rr = np.nan
        rr_row[fase] = round(rr, 4)
    sharpe_rows.append(sr_row)
    rachev_rows.append(rr_row)

table6_sharpe = pd.DataFrame(sharpe_rows).set_index('Strategy')
table6_rachev = pd.DataFrame(rachev_rows).set_index('Strategy')

print("TABLE 6 | Market Phase Performance")
print("\nPanel A: Sharpe Ratio")
display(table6_sharpe)
print("\nPanel B: Rachev Ratio (alpha=10%)")
display(table6_rachev)

---
## Ringkasan dan Temuan Utama

Berdasarkan seluruh eksperimen yang telah dijalankan, beberapa temuan utama dapat dirangkum sebagai berikut:


2. **Network Markowitz dengan γ tinggi** (γ ≥ 0.7) menunjukkan keunggulan dibandingkan NW tanpa penalti, mengonfirmasi bahwa penalti sentralitas efektif menekan eksposur ke aset yang sangat terhubung dalam jaringan.


4. Pada fase **Recovery**, hampir semua strategi memiliki Rachev Ratio > 1, yang mengindikasikan bahwa *upside tail* lebih besar dari *downside tail* — sebuah kondisi menguntungkan.


---


---

## Referensi

- Giudici, P., Sariev, A., & Toscani, G. (2020). *Network Models to Improve Automated Cryptocurrency Portfolio Management*. Risks, 8(3), 96. https://doi.org/10.3390/risks8030096

- Marchenko, V. A., & Pastur, L. A. (1967). Distribution of eigenvalues for some sets of random matrices. *Mathematics of the USSR-Sbornik*, 1(4), 457–483.

- Mantegna, R. N. (1999). Hierarchical structure in financial markets. *The European Physical Journal B*, 11(1), 193–197.